# LeetCode 514 — Freedom Trail bằng Python

- Link: https://leetcode.com/problems/freedom-trail/
- Độ khó: **Hard**
- Chủ đề: Dynamic Programming, Memoization, Khoảng cách trên vòng tròn, Binary Search

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def findRotateSteps(self, ring: str, key: str) -> int:

```

## 1. Mục tiêu học

1. Tính **khoảng cách trên vòng tròn** giữa hai vị trí: `min(|i - j|, n - |i - j|)`.
2. Nhận ra bài toán có **lựa chọn ở mỗi bước** (ký tự có thể xuất hiện nhiều lần trên vòng) và vì sao **tham lam chọn vị trí gần nhất là sai**.
3. Xây dựng **trạng thái DP** `(vị trí hiện tại trên vòng, số ký tự key đã gõ)` và cài cả **top-down (memo)** lẫn **bottom-up**.
4. Chứng minh và dùng tối ưu: **chỉ cần xét vị trí gần nhất theo chiều kim đồng hồ và ngược chiều** (lập luận "Lipschitz").
5. Kiểm thử chéo 4 cách với vét cạn trên dữ liệu ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Trong game Fallout 4, để mở cửa bạn phải xoay một **vòng chữ** `ring` để đánh vần một **từ khóa** `key`.

- Ban đầu, ký tự `ring[0]` nằm ở vị trí "12 giờ" (vị trí được chọn).
- Mỗi **bước**: xoay vòng **1 nấc** theo chiều kim đồng hồ hoặc ngược chiều, **hoặc** bấm nút giữa để "chốt" ký tự đang ở vị trí 12 giờ (bấm cũng tính 1 bước).
- Phải đánh vần lần lượt từng ký tự `key[0], key[1], …`: xoay để ký tự cần thiết về 12 giờ rồi bấm.

Trả về **số bước ít nhất** để đánh vần toàn bộ `key`.

**Ví dụ 1**

```text
Input:  ring = "godding", key = "gd"
Output: 4
Giải thích: 'g' đã ở 12 giờ -> bấm (1 bước).
            Xoay 2 nấc để một chữ 'd' về 12 giờ (2 bước), rồi bấm (1 bước).
            Tổng 1 + 2 + 1 = 4.
```

(Cụ thể: từ vị trí 0, ký tự `'d'` ở vị trí 2 và 3; vị trí 2 cách 2 nấc → 2 bước xoay.)

**Ví dụ 2**

```text
Input:  ring = "godding", key = "godding"
Output: 13
```

**Ràng buộc**

- `1 <= ring.length, key.length <= 100`
- `ring` và `key` chỉ gồm chữ thường tiếng Anh.
- **Đảm bảo** `key` luôn đánh vần được bằng `ring`.

Nhận xét: mọi lời giải đều tốn đúng `len(key)` lần bấm; phần cần tối ưu là **tổng số nấc xoay**.

In [1]:
ring, key = "godding", "gd"
n = len(ring)
print("vị trí các ký tự:", {c: [i for i, x in enumerate(ring) if x == c] for c in set(ring)})
print("bấm 'g' ở vị trí 0: 1 bước")
print("xoay từ 0 tới 2 ('d'):", min(abs(0 - 2), n - abs(0 - 2)), "bước + bấm 1 bước")

vị trí các ký tự: {'o': [1], 'n': [5], 'g': [0, 6], 'd': [2, 3], 'i': [4]}
bấm 'g' ở vị trí 0: 1 bước
xoay từ 0 tới 2 ('d'): 2 bước + bấm 1 bước


## 3. Kiến thức nền

### 3.1. Khoảng cách trên vòng tròn

Vòng có `n` vị trí `0..n-1`. Đi từ `i` tới `j`:
- Theo một chiều mất `|i - j|` nấc.
- Theo chiều kia mất `n - |i - j|` nấc.
- Khoảng cách = `min(|i - j|, n - |i - j|)`.

In [2]:
def circ(i, j, n):
    d = abs(i - j)
    return min(d, n - d)

n = 7
for i, j in [(0, 2), (0, 5), (6, 0), (3, 3)]:
    print(f"circ({i},{j}) trên vòng {n} = {circ(i, j, n)}")

circ(0,2) trên vòng 7 = 2
circ(0,5) trên vòng 7 = 2
circ(6,0) trên vòng 7 = 1
circ(3,3) trên vòng 7 = 0


### 3.2. Gom vị trí theo ký tự: `defaultdict(list)`

Ta cần biết nhanh "ký tự `c` xuất hiện ở những vị trí nào trên vòng".

In [3]:
from collections import defaultdict
pos = defaultdict(list)
for i, ch in enumerate("godding"):
    pos[ch].append(i)
print(dict(pos))

{'g': [0, 6], 'o': [1], 'd': [2, 3], 'i': [4], 'n': [5]}


### 3.3. Memoization với `functools.lru_cache`

Hàm `dp(i, p)` gọi lại với cùng tham số nhiều lần → `@lru_cache` ghi nhớ kết quả. Tham số phải hashable (số, chuỗi, tuple).

In [4]:
from functools import lru_cache

@lru_cache(maxsize=None)
def paths(r, c):
    """Số đường đi từ (0,0) tới (r,c) chỉ đi xuống/phải."""
    if r == 0 or c == 0:
        return 1
    return paths(r - 1, c) + paths(r, c - 1)

print(paths(10, 10), paths.cache_info())

184756 CacheInfo(hits=81, misses=120, maxsize=None, currsize=120)


### 3.4. `bisect` trên danh sách vị trí — tìm vị trí gần nhất hai phía

Danh sách vị trí của một ký tự đã **sắp xếp tăng** (do duyệt từ trái sang phải). Với vị trí hiện tại `p`:
- `k = bisect_left(lst, p)`: phần tử đầu tiên `>= p` là `lst[k % len(lst)]` (vòng lại nếu vượt).
- Phần tử cuối cùng `< p` là `lst[k - 1]` (Python cho phép `lst[-1]` khi `k = 0` — chính là vòng lại).

In [5]:
import bisect
lst = [2, 3, 6]          # vị trí của 'd' giả sử
for p in [0, 3, 4, 7 - 1]:
    k = bisect.bisect_left(lst, p)
    cw = lst[k % len(lst)]          # gần nhất theo chiều tăng chỉ số (có vòng)
    ccw = lst[k - 1]                # gần nhất theo chiều giảm chỉ số (có vòng)
    print(f"p={p}: gần nhất phía tăng = {cw}, phía giảm = {ccw}")

p=0: gần nhất phía tăng = 2, phía giảm = 6
p=3: gần nhất phía tăng = 3, phía giảm = 2
p=4: gần nhất phía tăng = 6, phía giảm = 3
p=6: gần nhất phía tăng = 6, phía giảm = 3


### 3.5. Giá trị vô cực `float('inf')` và `min` với `default`

Khởi tạo kết quả tối thiểu bằng `float('inf')`; `min(..., default=...)` tránh lỗi khi danh sách rỗng.

In [6]:
INF = float("inf")
print(min([5, 3, 8]), min([], default=INF), INF > 10**18)

3 inf True


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 2: `ring = "godding"` (vị trí: g:0,6 · o:1 · d:2,3 · i:4 · n:5), `key = "godding"`.

- `g`: đang ở 0, bấm → 1 bước. Vị trí 0.
- `o`: 0 → 1, xoay 1 + bấm 1 = 2. Vị trí 1.
- `d`: 1 → 2, xoay 1 + bấm = 2. Vị trí 2.
- `d`: 2 → 2 (hoặc 3), xoay 0 + bấm = 1. Vị trí 2.
- `i`: 2 → 4, xoay 2 + bấm = 3. Vị trí 4.
- `n`: 4 → 5, xoay 1 + bấm = 2. Vị trí 5.
- `g`: 5 → 6, xoay 1 + bấm = 2.

Tổng: `1 + 2 + 2 + 1 + 3 + 2 + 2 = 13`. ✔

Ở bước gõ `d` lần đầu có hai lựa chọn (vị trí 2 hoặc 3). Chọn gần nhất có vẻ hợp lý, nhưng **không phải lúc nào cũng đúng** — lựa chọn hôm nay ảnh hưởng chi phí ngày mai. Ví dụ phản chứng ở mục 13. Vì vậy cần xét **mọi** lựa chọn — và dùng DP để không bị bùng nổ tổ hợp.

In [7]:
ring, key = "godding", "godding"
n = len(ring)
p, total = 0, 0
for ch in key:
    best = min((i for i in range(n) if ring[i] == ch), key=lambda i: circ(p, i, n))
    total += circ(p, best, n) + 1
    print(f"gõ {ch}: {p} -> {best}, xoay {circ(p, best, n)}, cộng dồn {total}")
    p = best

gõ g: 0 -> 0, xoay 0, cộng dồn 1
gõ o: 0 -> 1, xoay 1, cộng dồn 3
gõ d: 1 -> 2, xoay 1, cộng dồn 5
gõ d: 2 -> 2, xoay 0, cộng dồn 6
gõ i: 2 -> 4, xoay 2, cộng dồn 9
gõ n: 4 -> 5, xoay 1, cộng dồn 11
gõ g: 5 -> 6, xoay 1, cộng dồn 13


## 5. Cách 1 — Brute Force: thử mọi lựa chọn (đệ quy không nhớ)

**Ý tưởng.** `go(i, p)` = số bước ít nhất để gõ `key[i:]` khi vòng đang ở vị trí `p`.
- `i == len(key)` → 0.
- Ngược lại: thử mọi vị trí `q` có `ring[q] == key[i]`: `circ(p, q) + 1 + go(i + 1, q)`; lấy min.

Không nhớ kết quả → số lời gọi có thể là `(số lần xuất hiện)^len(key)`.

In [8]:
def rotate_brute(ring, key):
    n = len(ring)
    pos = defaultdict(list)
    for i, ch in enumerate(ring):
        pos[ch].append(i)

    def go(i, p):
        if i == len(key):
            return 0
        return min(circ(p, q, n) + 1 + go(i + 1, q) for q in pos[key[i]])

    return go(0, 0)

print(rotate_brute("godding", "gd"), rotate_brute("godding", "godding"))
assert rotate_brute("godding", "gd") == 4
assert rotate_brute("godding", "godding") == 13

4 13


**Độ phức tạp:** nếu mỗi ký tự xuất hiện `t` lần, số nhánh `t^m` (`m = len(key)`). Với `ring = "aaaa…"` (100 chữ `a`) và `key` dài 100 → `100^100`. Bất khả thi.

**Nút thắt:** cùng một trạng thái `(i, p)` bị tính lại vô số lần. Nhưng chỉ có `m × n ≤ 10^4` trạng thái khác nhau!

In [9]:
import time
calls = 0
def count_calls(ring, key):
    global calls
    n = len(ring)
    pos = defaultdict(list)
    for i, ch in enumerate(ring):
        pos[ch].append(i)
    def go(i, p):
        global calls
        calls += 1
        if i == len(key):
            return 0
        return min(circ(p, q, n) + 1 + go(i + 1, q) for q in pos[key[i]])
    return go(0, 0)

for m in [4, 6, 8]:
    calls = 0
    count_calls("abababab", "ab" * (m // 2))
    print(f"key dài {m}: {calls} lời gọi đệ quy")

key dài 4: 341 lời gọi đệ quy
key dài 6: 5461 lời gọi đệ quy
key dài 8: 87381 lời gọi đệ quy


## 6. Bước đột phá

**Insight 1 — Trạng thái chỉ cần `(i, p)`.** Chi phí tương lai chỉ phụ thuộc vào **vị trí hiện tại** `p` và **số ký tự đã gõ** `i`; không phụ thuộc vào cách ta tới đó. → DP với `m × n` trạng thái, mỗi trạng thái thử ≤ `n` lựa chọn → `O(m · n²)`.

Công thức truy hồi:

```text
dp(i, p) = 0                                              nếu i == m
dp(i, p) = min over q với ring[q] == key[i] của  circ(p, q) + 1 + dp(i+1, q)
Đáp án: dp(0, 0)
```

**Insight 2 — Chỉ cần 2 ứng viên.** Từ `p`, trong các vị trí `q` có ký tự `key[i]`, chỉ cần xét:
- `x` = vị trí gần nhất khi đi **theo chiều tăng chỉ số** (có vòng),
- `y` = vị trí gần nhất khi đi **theo chiều giảm chỉ số**.

Vì sao? Gọi `f(q) = dp(i+1, q)`. Hàm `f` là **1-Lipschitz** theo khoảng cách vòng: `f(a) <= circ(a, b) + f(b)` (từ `a` có thể xoay sang `b` rồi làm như từ `b`). Với một ứng viên `q` khác `x, y`: đường xoay ngắn nhất từ `p` tới `q` (theo một chiều nào đó) **đi ngang qua** `x` hoặc `y` — giả sử `x`. Khi đó `circ(p, q) = circ(p, x) + (đoạn từ x tới q theo cùng chiều) >= circ(p, x) + circ(x, q)`, nên

`circ(p, x) + f(x) <= circ(p, x) + circ(x, q) + f(q) <= circ(p, q) + f(q)`.

Tức là dừng ở `x` không bao giờ tệ hơn dừng ở `q`. → mỗi trạng thái chỉ 2 lựa chọn → `O(m · n)` (hoặc `O(m log n)` trạng thái đạt được với bisect).

**Quy tắc:** **nếu** `ring[p] == key[i]` **thì** `x = y = p` (không cần xoay).

In [10]:
# Minh họa Insight 1: số trạng thái nhỏ
ring, key = "godding", "godding"
print("số trạng thái (i, p):", len(key) * len(ring))
# Minh họa Insight 2: từ p=0 tìm 'd' gần nhất hai phía trong "godding"
lst = [2, 3]; p = 0
k = bisect.bisect_left(lst, p)
print("gần nhất phía tăng:", lst[k % len(lst)], "| phía giảm:", lst[k - 1])

số trạng thái (i, p): 49
gần nhất phía tăng: 2 | phía giảm: 3


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Top-down DP (memoization)

In [11]:
def rotate_memo(ring, key):
    n, m = len(ring), len(key)
    pos = defaultdict(list)
    for i, ch in enumerate(ring):
        pos[ch].append(i)

    @lru_cache(maxsize=None)
    def dp(i, p):
        if i == m:
            return 0
        return min(circ(p, q, n) + 1 + dp(i + 1, q) for q in pos[key[i]])

    return dp(0, 0)

print(rotate_memo("godding", "gd"), rotate_memo("godding", "godding"))

4 13


**Code nộp LeetCode (Cách 2):**

In [12]:
class SolutionMemo:
    def findRotateSteps(self, ring: str, key: str) -> int:
        n, m = len(ring), len(key)
        pos = defaultdict(list)
        for i, ch in enumerate(ring):
            pos[ch].append(i)

        @lru_cache(maxsize=None)
        def dp(i: int, p: int) -> int:
            if i == m:
                return 0
            best = float("inf")
            for q in pos[key[i]]:
                d = abs(p - q)
                best = min(best, min(d, n - d) + 1 + dp(i + 1, q))
            return best

        return dp(0, 0)

print(SolutionMemo().findRotateSteps("godding", "godding"))

13


**Độ phức tạp Cách 2:** `m · n` trạng thái × ≤ `n` lựa chọn → Time `O(m · n²)` = `10^6` với giới hạn đề. Space `O(m · n)` cho memo + `O(m)` stack đệ quy.

### 7.2. Cách 3 — Bottom-up DP chỉ trên các vị trí có ký tự đúng

Gõ từ trái sang phải. Gọi `cur[q]` = chi phí nhỏ nhất để gõ xong `key[0..i]` và **dừng ở `q`** (với `ring[q] == key[i]`). Chuyển:

```text
cur_next[q] = min over p in cur của  cur[p] + circ(p, q) + 1        (ring[q] == key[i+1])
```

Khởi tạo: "trước khi gõ" ta ở vị trí 0 với chi phí 0. Đáp án: `min(cur.values())` sau ký tự cuối. Ta chỉ lưu một "lớp" (dict) → Space `O(n)`.

In [13]:
def rotate_bottom_up(ring, key):
    n = len(ring)
    pos = defaultdict(list)
    for i, ch in enumerate(ring):
        pos[ch].append(i)
    cur = {0: 0}                                  # vị trí -> chi phí tốt nhất
    for ch in key:
        nxt = {}
        for q in pos[ch]:
            nxt[q] = min(cost + circ(p, q, n) for p, cost in cur.items()) + 1
        cur = nxt
    return min(cur.values())

print(rotate_bottom_up("godding", "gd"), rotate_bottom_up("godding", "godding"))

4 13


**Code nộp LeetCode (Cách 3):**

In [14]:
class SolutionBottomUp:
    def findRotateSteps(self, ring: str, key: str) -> int:
        n = len(ring)
        pos = defaultdict(list)
        for i, ch in enumerate(ring):
            pos[ch].append(i)
        cur = {0: 0}
        for ch in key:
            nxt = {}
            for q in pos[ch]:
                best = float("inf")
                for p, cost in cur.items():
                    d = abs(p - q)
                    best = min(best, cost + min(d, n - d))
                nxt[q] = best + 1
            cur = nxt
        return min(cur.values())

print(SolutionBottomUp().findRotateSteps("godding", "gd"))

4


**Độ phức tạp Cách 3:** mỗi ký tự key: `|pos[key[i-1]]| × |pos[key[i]]| <= n²` → Time `O(m · n²)` (thường nhỏ hơn nhiều), Space `O(n)`.

### 7.3. Cách 4 — DP chỉ xét 2 ứng viên gần nhất (bisect)

Dùng Insight 2: ở mỗi trạng thái `(i, p)` chỉ thử vị trí gần nhất phía tăng và phía giảm.

In [15]:
def rotate_two_candidates(ring, key):
    n, m = len(ring), len(key)
    pos = defaultdict(list)
    for i, ch in enumerate(ring):
        pos[ch].append(i)

    @lru_cache(maxsize=None)
    def dp(i, p):
        if i == m:
            return 0
        lst = pos[key[i]]
        k = bisect.bisect_left(lst, p)
        cands = {lst[k % len(lst)], lst[k - 1]}     # gần nhất hai phía (có vòng)
        return min(circ(p, q, n) + 1 + dp(i + 1, q) for q in cands)

    return dp(0, 0)

print(rotate_two_candidates("godding", "gd"), rotate_two_candidates("godding", "godding"))

4 13


**Code nộp LeetCode (Cách 4):**

In [16]:
class SolutionTwo:
    def findRotateSteps(self, ring: str, key: str) -> int:
        return rotate_two_candidates(ring, key)

print(SolutionTwo().findRotateSteps("godding", "godding"))

13


**Độ phức tạp Cách 4:** mỗi trạng thái 2 lựa chọn, tìm bằng `bisect` `O(log n)`. Số trạng thái **đạt được** từ `(0, 0)` tối đa `2^i` ở tầng `i` nhưng cũng ≤ `n` → Time `O(m · n · log n)` xấu nhất, thực tế rất ít. Space `O(m · n)`.

## 8. TRACE từng bước

Trace **Cách 3 (bottom-up)** trên `ring = "godding"`, `key = "gd"`:

| bước | ký tự | vị trí ứng viên `q` | chi phí `cur[q]` |
|---|---|---|---|
| khởi tạo | – | 0 | 0 |
| 1 | g | 0, 6 | `0+0+1 = 1`, `0+1+1 = 2` |
| 2 | d | 2, 3 | `min(1+2, 2+3)+1 = 4`, `min(1+3, 2+3)+1 = 5` |

Đáp án `min(4, 5) = 4`.

In [17]:
def trace_bottom_up(ring, key):
    n = len(ring)
    pos = defaultdict(list)
    for i, ch in enumerate(ring):
        pos[ch].append(i)
    cur = {0: 0}
    print(f"khởi tạo: cur = {cur}")
    for step, ch in enumerate(key, 1):
        nxt = {}
        for q in pos[ch]:
            options = {p: cost + circ(p, q, n) for p, cost in cur.items()}
            bp = min(options, key=options.get)
            nxt[q] = options[bp] + 1
            print(f"  bước {step} '{ch}': tới q={q}: lựa chọn từ {options} -> tốt nhất từ p={bp}, +1 bấm = {nxt[q]}")
        cur = nxt
        print(f"bước {step}: cur = {cur}")
    print("Đáp án:", min(cur.values()))
    return min(cur.values())

trace_bottom_up("godding", "gd")

khởi tạo: cur = {0: 0}
  bước 1 'g': tới q=0: lựa chọn từ {0: 0} -> tốt nhất từ p=0, +1 bấm = 1
  bước 1 'g': tới q=6: lựa chọn từ {0: 1} -> tốt nhất từ p=0, +1 bấm = 2
bước 1: cur = {0: 1, 6: 2}
  bước 2 'd': tới q=2: lựa chọn từ {0: 3, 6: 5} -> tốt nhất từ p=0, +1 bấm = 4
  bước 2 'd': tới q=3: lựa chọn từ {0: 4, 6: 5} -> tốt nhất từ p=0, +1 bấm = 5
bước 2: cur = {2: 4, 3: 5}
Đáp án: 4


4

Trace **Cách 4** — in các trạng thái `(i, p)` thực sự được tính và 2 ứng viên mỗi lần:

In [18]:
def trace_two(ring, key):
    n, m = len(ring), len(key)
    pos = defaultdict(list)
    for i, ch in enumerate(ring):
        pos[ch].append(i)
    memo = {}
    def dp(i, p, depth=0):
        if i == m:
            return 0
        if (i, p) in memo:
            return memo[(i, p)]
        lst = pos[key[i]]
        k = bisect.bisect_left(lst, p)
        cands = sorted({lst[k % len(lst)], lst[k - 1]})
        res = min(circ(p, q, n) + 1 + dp(i + 1, q, depth + 1) for q in cands)
        print(f"{'  ' * depth}dp(i={i}, p={p}) gõ '{key[i]}': ứng viên {cands} -> {res}")
        memo[(i, p)] = res
        return res
    return dp(0, 0)

trace_two("godding", "godding")

            dp(i=6, p=5) gõ 'g': ứng viên [0, 6] -> 2
          dp(i=5, p=4) gõ 'n': ứng viên [5] -> 4
        dp(i=4, p=2) gõ 'i': ứng viên [4] -> 7
        dp(i=4, p=3) gõ 'i': ứng viên [4] -> 6
      dp(i=3, p=2) gõ 'd': ứng viên [2, 3] -> 8
      dp(i=3, p=3) gõ 'd': ứng viên [2, 3] -> 7
    dp(i=2, p=1) gõ 'd': ứng viên [2, 3] -> 10
  dp(i=1, p=0) gõ 'o': ứng viên [1] -> 12
  dp(i=1, p=6) gõ 'o': ứng viên [1] -> 13
dp(i=0, p=0) gõ 'g': ứng viên [0, 6] -> 13


13

## 9. Code nộp LeetCode

Bản nộp chính: bottom-up trên các vị trí đúng ký tự (không đệ quy, dễ hiểu, `O(m · n²)` đủ nhanh).

In [19]:
from collections import defaultdict

class Solution:
    def findRotateSteps(self, ring: str, key: str) -> int:
        n = len(ring)
        pos = defaultdict(list)                 # ký tự -> các vị trí trên vòng
        for i, ch in enumerate(ring):
            pos[ch].append(i)

        cur = {0: 0}                            # vị trí đang ở 12 giờ -> chi phí nhỏ nhất
        for ch in key:
            nxt = {}
            for q in pos[ch]:
                best = float("inf")
                for p, cost in cur.items():
                    d = abs(p - q)
                    best = min(best, cost + min(d, n - d))
                nxt[q] = best + 1               # +1 cho lần bấm nút
            cur = nxt
        return min(cur.values())

sol = Solution()
assert sol.findRotateSteps("godding", "gd") == 4
assert sol.findRotateSteps("godding", "godding") == 13
print("Solution OK")

Solution OK


## 10. Vì sao thuật toán đúng

**Cấu trúc con tối ưu.** Mọi cách đánh vần là một dãy vị trí `q_0, q_1, …, q_{m-1}` với `ring[q_i] == key[i]`. Chi phí = `circ(0, q_0) + Σ circ(q_{i-1}, q_i) + m`. (Giữa hai lần bấm, xoay theo đường ngắn nhất là tối ưu; xoay vòng vo chỉ tốn thêm.) Chi phí tách được theo từng cặp liên tiếp → phần đuôi chỉ phụ thuộc vào `q_{i-1}`.

**Bottom-up.** Bất biến: sau khi xử lý `key[0..i]`, `cur[q]` = chi phí nhỏ nhất của mọi dãy `q_0..q_i` với `q_i = q`. Khởi tạo đúng (`{0: 0}` là trạng thái trước khi gõ). Bước chuyển lấy min trên mọi `q_{i-1}` có thể — đúng định nghĩa. Đáp án là min trên mọi `q_{m-1}`.

**Hai ứng viên (Cách 4).** Gọi `f(q) = dp(i+1, q)`. Ta có `f(a) <= circ(a, b) + f(b)` vì từ `a` có thể xoay tới `b` trước khi làm theo kế hoạch tối ưu từ `b` (lần xoay đầu tiên của kế hoạch từ `b` cộng thêm `circ(a, b)`; bất đẳng thức tam giác trên vòng). Với ứng viên `q ∉ {x, y}`, đường ngắn nhất từ `p` tới `q` đi theo một chiều và **gặp** `x` (nếu chiều tăng) hoặc `y` (nếu chiều giảm) trước, nên `circ(p, q) = circ(p, x) + (đoạn còn lại) >= circ(p, x) + circ(x, q)`. Kết hợp: `circ(p, x) + f(x) <= circ(p, q) + f(q)`. Vậy bỏ `q` không làm mất tối ưu.

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Brute | `O(t^m)` | `O(m)` |
| 2. Memo | `O(m · n²)` | `O(m · n)` |
| 3. Bottom-up | `O(m · n²)` (thực tế `Σ |pos[key[i-1]]|·|pos[key[i]]|`) | `O(n)` |
| 4. Hai ứng viên | `O(m · n · log n)` xấu nhất | `O(m · n)` |

In [20]:
import random
random.seed(514)
ring_big = "".join(random.choice("ab") for _ in range(100))
key_big = "".join(random.choice("ab") for _ in range(100))
for name, f in [("memo", rotate_memo), ("bottom_up", rotate_bottom_up), ("two", rotate_two_candidates), ("Solution", Solution().findRotateSteps)]:
    t = time.perf_counter()
    r = f(ring_big, key_big)
    print(f"{name:<10}: {time.perf_counter() - t:.4f}s -> {r}")

memo      : 0.0365s -> 155
bottom_up : 0.0219s -> 155
two       : 0.0031s -> 155
Solution  : 0.0167s -> 155


## 12. Bộ test

In [21]:
methods = {
    "brute": rotate_brute,
    "memo": rotate_memo,
    "bottom_up": rotate_bottom_up,
    "two": rotate_two_candidates,
    "SolMemo": SolutionMemo().findRotateSteps,
    "SolBottomUp": SolutionBottomUp().findRotateSteps,
    "SolTwo": SolutionTwo().findRotateSteps,
    "Solution": Solution().findRotateSteps,
}
cases = [
    ("godding", "gd", 4),
    ("godding", "godding", 13),
    ("a", "a", 1),                     # vòng 1 ký tự
    ("a", "aaaa", 4),                  # chỉ bấm
    ("abc", "c", 2),                   # xoay ngược 1 nấc
    ("abcde", "eda", 7),               # e: xoay 1 + bấm; d: 1 + bấm; a: 2 + bấm
    ("aaaaa", "aaa", 3),               # mọi vị trí đều đúng ký tự
]
for ring, key, exp in cases:
    for name, f in methods.items():
        if name == "brute" and len(key) > 12:
            continue
        got = f(ring, key)
        assert got == exp, (name, ring, key, got, exp)
print("Ví dụ + edge case: OK")

Ví dụ + edge case: OK


In [22]:
random.seed(2026)
for trial in range(600):
    alphabet = random.choice(["ab", "abc", "abcd"])
    ring = "".join(random.choice(alphabet) for _ in range(random.randint(1, 9)))
    key = "".join(random.choice(ring) for _ in range(random.randint(1, 7)))   # đảm bảo đánh vần được
    exp = rotate_brute(ring, key)
    for name, f in methods.items():
        got = f(ring, key)
        assert got == exp, (name, ring, key, got, exp)
print("600 test ngẫu nhiên (seed 2026) khớp brute force")

600 test ngẫu nhiên (seed 2026) khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tham lam chọn vị trí gần nhất

**Code sai:** mỗi bước xoay tới vị trí gần nhất có ký tự cần gõ.

**Vì sao sai:** lựa chọn gần nhất bây giờ có thể đẩy ta ra xa các ký tự tiếp theo. Ta tìm phản ví dụ tự động:

In [23]:
def rotate_greedy(ring, key):
    n, p, total = len(ring), 0, 0
    for ch in key:
        q = min((i for i in range(n) if ring[i] == ch), key=lambda i: circ(p, i, n))
        total += circ(p, q, n) + 1
        p = q
    return total

random.seed(1)
found = None
for _ in range(20000):
    ring = "".join(random.choice("abc") for _ in range(random.randint(3, 8)))
    key = "".join(random.choice(ring) for _ in range(random.randint(2, 4)))
    if rotate_greedy(ring, key) != rotate_brute(ring, key):
        found = (ring, key); break
print("phản ví dụ:", found, "| tham lam:", rotate_greedy(*found), "| đúng:", rotate_brute(*found))

phản ví dụ: ('abcca', 'ca') | tham lam: 6 | đúng: 5


**Code đúng:** DP xét mọi vị trí (hoặc ít nhất cả hai ứng viên hai phía).

### Lỗi 2 — Quên cộng bước bấm nút

**Code sai:** chỉ cộng số nấc xoay → ví dụ 1 ra 2 thay vì 4.

**Code đúng:** cộng `+1` cho mỗi ký tự (hoặc cộng `len(key)` một lần ở cuối).

In [24]:
def rotate_no_press(ring, key):
    n = len(ring)
    pos = defaultdict(list)
    for i, ch in enumerate(ring):
        pos[ch].append(i)
    cur = {0: 0}
    for ch in key:
        cur = {q: min(c + circ(p, q, n) for p, c in cur.items()) for q in pos[ch]}   # thiếu +1
    return min(cur.values())

print("thiếu bấm:", rotate_no_press("godding", "gd"), "| đúng:", Solution().findRotateSteps("godding", "gd"))
print("sửa nhanh: + len(key) =", rotate_no_press("godding", "gd") + len("gd"))

thiếu bấm: 2 | đúng: 4
sửa nhanh: + len(key) = 4


### Lỗi 3 — Khoảng cách chỉ theo một chiều

**Code sai:** `abs(p - q)` hoặc `(q - p) % n` → quên rằng có thể xoay theo chiều ngắn hơn.

In [25]:
def rotate_one_way(ring, key):
    n = len(ring)
    pos = defaultdict(list)
    for i, ch in enumerate(ring):
        pos[ch].append(i)
    cur = {0: 0}
    for ch in key:
        cur = {q: min(c + abs(p - q) for p, c in cur.items()) + 1 for q in pos[ch]}   # SAI: không vòng
    return min(cur.values())

print("một chiều:", rotate_one_way("abcde", "e"), "| đúng:", Solution().findRotateSteps("abcde", "e"))

một chiều: 5 | đúng: 2


**Code đúng:** `min(d, n - d)` với `d = abs(p - q)`.

### Lỗi 4 — Bắt đầu sai vị trí / sai trạng thái khởi tạo

**Code sai:** khởi tạo `cur = {q: 1 for q in pos[key[0]]}` (bỏ qua chi phí xoay tới ký tự đầu tiên).

**Code đúng:** bắt đầu từ `cur = {0: 0}` (vị trí 0 ở 12 giờ) rồi xử lý `key[0]` như mọi ký tự khác.

## 14. Mẹo & nhận dạng pattern

- "Thực hiện dãy hành động theo thứ tự, mỗi hành động có **nhiều cách**, chi phí phụ thuộc **vị trí hiện tại**" → DP với trạng thái `(chỉ số hành động, vị trí)`.
- Vòng tròn → khoảng cách `min(d, n - d)`; danh sách vị trí + `bisect` với chỉ số `% len` để vòng lại.
- Tham lam "gần nhất" thường sai trong các bài có tương lai — **tìm phản ví dụ tự động** bằng vét cạn.
- Hàm chi phí tương lai 1-Lipschitz theo khoảng cách → chỉ cần xét **ứng viên gần nhất mỗi phía**.
- Các bài cùng khuôn: LC 1320 Minimum Distance to Type a Word Using Two Fingers, LC 72 Edit Distance (DP theo chỉ số).

## 15. Biến thể

1. **Chỉ được xoay một chiều:** khoảng cách là `(q - p) % n`.
2. **Chi phí xoay khác nhau theo chiều** (chiều kim đồng hồ `a`, ngược chiều `b` mỗi nấc): `min(a·d1, b·d2)`.
3. **Trả về dãy thao tác** (vết): lưu `parent` trong bottom-up.
4. **Hai vòng song song / hai ngón tay** (LC 1320): trạng thái thêm một chiều.

Demo biến thể 2:

In [26]:
def rotate_weighted(ring, key, a, b):
    n = len(ring)
    pos = defaultdict(list)
    for i, ch in enumerate(ring):
        pos[ch].append(i)
    cur = {0: 0}
    for ch in key:
        cur = {q: min(c + min(a * ((q - p) % n), b * ((p - q) % n)) for p, c in cur.items()) + 1 for q in pos[ch]}
    return min(cur.values())

print(rotate_weighted("godding", "gd", 1, 1), rotate_weighted("godding", "gd", 1, 5), rotate_weighted("godding", "gd", 5, 1))
assert rotate_weighted("godding", "godding", 1, 1) == 13

4 4 6


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | Thử mọi vị trí, không nhớ | `O(t^m)` | `O(m)` | Đối chiếu |
| 2. Memo | `dp(i, p)` + `lru_cache` | `O(m n²)` | `O(m n)` | Viết nhanh từ truy hồi |
| 3. Bottom-up | Một lớp dict theo vị trí | `O(m n²)` | `O(n)` | Lời giải nộp, tiết kiệm bộ nhớ |
| 4. Hai ứng viên | Chỉ gần nhất hai phía (bisect) | `O(m n log n)` | `O(m n)` | Khi `n` lớn |

## 17. Bài tập

**Bài 1 (đoán output).** `circ(1, 6, 7)` bằng bao nhiêu?

**Bài 2 (tính tay).** `ring = "abcb"`, `key = "bc"`. Đáp án là bao nhiêu? Gõ `b` ở vị trí nào thì tốt?

**Bài 3 (điền TODO).** Hoàn thiện bước chuyển bottom-up.

In [27]:
def rotate_todo(ring, key):
    n = len(ring)
    pos = defaultdict(list)
    for i, ch in enumerate(ring):
        pos[ch].append(i)
    cur = {0: 0}
    for ch in key:
        nxt = {}
        for q in pos[ch]:
            # TODO: nxt[q] = min_{p} (cur[p] + circ(p, q, n)) + 1
            pass
        cur = nxt
    return min(cur.values())

# Không gọi khi chạy notebook. Tự kiểm tra:
# assert rotate_todo("godding", "gd") == 4

**Bài 4 (tự viết lại).** Viết lại Cách 2 (memo) không nhìn code.

**Bài 5 (mini challenge).** Viết `rotate_with_path(ring, key)` trả về `(số bước, danh sách vị trí q_0..q_{m-1})` của một lời giải tối ưu.

In [28]:
def rotate_with_path_todo(ring, key):
    # TODO
    raise NotImplementedError

# Không gọi khi chạy notebook.

## 18. Gợi ý 3 mức

**Bài 1–2**
- Mức 1: `d = |1 - 6| = 5`, `n - d = 2`.
- Mức 2: Bài 2: `b` ở vị trí 1 và 3, `c` ở vị trí 2.
- Mức 3: Bài 2: xoay 1 tới vị trí 1 (hoặc vị trí 3, cũng 1 nấc), bấm; xoay 1 tới `c`, bấm → 4.

**Bài 3**
- Mức 1: Duyệt `p, cost in cur.items()`.
- Mức 2: Lấy min biểu thức `cost + circ(p, q, n)`.
- Mức 3: Đừng quên `+ 1`.

**Bài 4**
- Mức 1: `@lru_cache` trên `dp(i, p)`.
- Mức 2: Cơ sở `i == len(key)` → 0.
- Mức 3: Đáp án `dp(0, 0)`.

**Bài 5**
- Mức 1: Lưu thêm `parent[i][q] = p` tốt nhất.
- Mức 2: Sau cùng chọn `q` có chi phí nhỏ nhất, lần ngược `parent`.
- Mức 3: Kiểm tra: tính lại chi phí từ dãy vị trí phải bằng số bước.

## 19. Đáp án tham khảo

In [29]:
# Bài 1, 2
assert circ(1, 6, 7) == 2
assert Solution().findRotateSteps("abcb", "bc") == 4

# Bài 3
def rotate_sol(ring, key):
    n = len(ring)
    pos = defaultdict(list)
    for i, ch in enumerate(ring):
        pos[ch].append(i)
    cur = {0: 0}
    for ch in key:
        cur = {q: min(c + circ(p, q, n) for p, c in cur.items()) + 1 for q in pos[ch]}
    return min(cur.values())
assert rotate_sol("godding", "gd") == 4

# Bài 4
def rotate_memo_ex(ring, key):
    n = len(ring)
    @lru_cache(maxsize=None)
    def dp(i, p):
        if i == len(key):
            return 0
        return min(circ(p, q, n) + 1 + dp(i + 1, q) for q in range(n) if ring[q] == key[i])
    return dp(0, 0)

random.seed(21)
for _ in range(300):
    ring = "".join(random.choice("abc") for _ in range(random.randint(1, 8)))
    key = "".join(random.choice(ring) for _ in range(random.randint(1, 6)))
    assert rotate_sol(ring, key) == rotate_memo_ex(ring, key) == rotate_brute(ring, key)
print("Bài 1–4 OK")

Bài 1–4 OK


In [30]:
# Bài 5
def rotate_with_path(ring, key):
    n = len(ring)
    pos = defaultdict(list)
    for i, ch in enumerate(ring):
        pos[ch].append(i)
    cur = {0: 0}
    parents = []
    for ch in key:
        nxt, par = {}, {}
        for q in pos[ch]:
            bp = min(cur, key=lambda p: cur[p] + circ(p, q, n))
            nxt[q] = cur[bp] + circ(bp, q, n) + 1
            par[q] = bp
        parents.append(par)
        cur = nxt
    q = min(cur, key=cur.get)
    best = cur[q]
    path = []
    for par in reversed(parents):
        path.append(q)
        q = par[q]
    return best, path[::-1]

def cost_of(ring, key, path):
    n = len(ring); p = 0; total = 0
    for ch, q in zip(key, path):
        assert ring[q] == ch
        total += circ(p, q, n) + 1; p = q
    return total

print(rotate_with_path("godding", "godding"))
random.seed(22)
for _ in range(300):
    ring = "".join(random.choice("abc") for _ in range(random.randint(1, 8)))
    key = "".join(random.choice(ring) for _ in range(random.randint(1, 6)))
    best, path = rotate_with_path(ring, key)
    assert best == rotate_brute(ring, key) == cost_of(ring, key, path)
print("Bài 5 OK")

(13, [0, 1, 2, 2, 4, 5, 6])
Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template DP theo vị trí trên vòng:**

```python
pos = defaultdict(list)
for i, ch in enumerate(ring): pos[ch].append(i)
cur = {0: 0}
for ch in key:
    cur = {q: min(c + min(abs(p-q), n-abs(p-q)) for p, c in cur.items()) + 1
           for q in pos[ch]}
return min(cur.values())
```

**Checklist trước khi submit:**

- [ ] Khoảng cách vòng `min(d, n - d)`.
- [ ] Cộng `+1` cho mỗi lần bấm.
- [ ] Khởi tạo từ vị trí `0`, chi phí `0`.
- [ ] Không dùng tham lam "gần nhất".
- [ ] Test vòng 1 ký tự, ký tự lặp liên tiếp trong key, key dài 100.
- [ ] Đối chiếu brute force trên input nhỏ ngẫu nhiên.